# 05 - NBC News Data Repair

Backfill notebook for recommendations from combined exploration.

This pass repairs missing `author` values in NBC data

## 0. Imports and Data Load

In [27]:
import json
import re
from typing import Any, Dict, List, Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [28]:
fox = pd.read_csv("../data/raw/fox_scraped_all.csv")
nbc = pd.read_csv("../data/raw/nbc_scraped_all.csv")

print(f"FOX rows: {len(fox)}")
print(f"NBC rows: {len(nbc)}")
print(f"NBC missing authors before repair: {nbc['author'].isna().sum()}")

FOX rows: 2000
NBC rows: 1805
NBC missing authors before repair: 96


## 1. Scope the Repair Target

From earlier exploration, we identified 80 NBC rows where live-update URLs correspond to missing author data. This section recreates that exact scope and normalizes URLs for consistent page-level processing.

In [29]:
live_mask = (
    nbc["title"].notna()
    & nbc["author"].isna()
    & nbc["url"].str.contains("live-updates", na=False)
)

live_source_rows = nbc.loc[live_mask].copy()
print(f"Live-update source rows: {len(live_source_rows)}")
assert len(live_source_rows) == 80, "Expected exactly 80 live-update source rows."


def normalize_live_url(url: str) -> str:
    if pd.isna(url):
        return url
    clean = str(url).split("?")[0].rstrip("/")
    clean = re.sub(r"/rcrd\d+$", "", clean)
    return clean


live_source_rows["normalized_url"] = live_source_rows["url"].map(normalize_live_url)
print(f"Unique normalized live URLs: {live_source_rows['normalized_url'].nunique()}")
live_source_rows[["url", "normalized_url", "topic", "title"]].head()

Live-update source rows: 80
Unique normalized live URLs: 64


,url,normalized_url,topic,title
12,https://www.nbcnews.com/politics/supreme-court...,https://www.nbcnews.com/politics/supreme-court...,politics,Trump has some immunity in D.C. election inter...
25,https://www.nbcnews.com/politics/2024-election...,https://www.nbcnews.com/politics/2024-election...,politics,Coalition to March on the DNCâs protest ends...
28,https://www.nbcnews.com/politics/donald-trump/...,https://www.nbcnews.com/politics/donald-trump/...,politics,Trump indictment: Former president faces 37 co...
68,https://www.nbcnews.com/news/world/live-blog/i...,https://www.nbcnews.com/news/world/live-blog/i...,news,IDF has been securing aid convoys for 4 nights...
69,https://www.nbcnews.com/news/world/live-blog/i...,https://www.nbcnews.com/news/world/live-blog/i...,news,"Gaza hits 96-hour telecom blackout, a new record"


## 2. Scrape Author/Contributor Metadata

For live-update pages, bylines are often present in metadata (`<meta>` tags or JSON-LD) rather than in-page article text.

In [30]:
SESSION = requests.Session()
SESSION.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/125.0.0.0 Safari/537.36"
        )
    }
)


def clean_text(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    text = re.sub(r"\s+", " ", value).strip()
    return text if text else None


def build_jsonld_live_card_index(soup: BeautifulSoup) -> Dict[str, Dict[str, Any]]:
    """Recursively walk JSON-LD and attach metadata to live card ids (rcrd...)."""
    card_index: Dict[str, Dict[str, Any]] = {}
    scripts = soup.find_all("script", attrs={"type": "application/ld+json"})

    for script in scripts:
        raw = script.string or script.get_text(strip=True)
        if not raw:
            continue

        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            continue

        stack = [data]
        while stack:
            current = stack.pop()

            if isinstance(current, dict):
                page_ref = str(
                    current.get("url")
                    or current.get("@id")
                    or (current.get("mainEntityOfPage") or {}).get("@id", "")
                )
                match = re.search(r"#(rcrd\d+)$", page_ref)

                if match:
                    card_id = match.group(1)
                    blob = card_index.setdefault(card_id, {})

                    author_field = current.get("author")
                    names: List[str] = []

                    if isinstance(author_field, list):
                        for entry in author_field:
                            if isinstance(entry, dict) and entry.get("name"):
                                names.append(str(entry["name"]))
                            elif isinstance(entry, str):
                                names.append(entry)
                    elif isinstance(author_field, dict) and author_field.get("name"):
                        names.append(str(author_field["name"]))
                    elif isinstance(author_field, str):
                        names.append(author_field)

                    cleaned_authors: List[str] = []
                    seen_names = set()
                    for name in names:
                        normalized = clean_text(name)
                        if normalized and normalized.lower() not in seen_names:
                            cleaned_authors.append(normalized)
                            seen_names.add(normalized.lower())

                    if cleaned_authors and "authors" not in blob:
                        blob["authors"] = cleaned_authors

                    ts = clean_text(current.get("datePublished") or current.get("dateModified"))
                    if ts and "datetime_posted" not in blob:
                        blob["datetime_posted"] = ts

                    headline = clean_text(current.get("headline"))
                    if headline and "headline" not in blob:
                        blob["headline"] = headline

                for value in current.values():
                    if isinstance(value, (dict, list)):
                        stack.append(value)

            elif isinstance(current, list):
                for item in current:
                    if isinstance(item, (dict, list)):
                        stack.append(item)

    return card_index


def parse_live_blog_cards(source_row: Dict[str, str]) -> List[Dict[str, str]]:
    url = source_row["normalized_url"]
    response = SESSION.get(url, timeout=25)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    main_title = clean_text(source_row.get("title"))
    card_nodes = soup.find_all("div", id=re.compile(r"^rcrd\d+"))
    jsonld_index = build_jsonld_live_card_index(soup)

    parsed = []
    for card in card_nodes:
        card_id = card.get("id")
        jmeta = jsonld_index.get(card_id, {}) if card_id else {}

        pubat = card.get("data-pubat") or jmeta.get("datetime_posted")

        headline_node = card.find("h2")
        subtitle = clean_text(headline_node.get_text(" ", strip=True) if headline_node else None)
        if not subtitle and jmeta.get("headline"):
            subtitle = clean_text(jmeta["headline"])

        authors = []
        seen = set()

        # First pass: exact name nodes when available.
        author_nodes = card.select("span[data-testid='author-name']")
        for node in author_nodes:
            name = clean_text(node.get_text(" ", strip=True))
            if name:
                key = name.lower()
                if key not in seen:
                    authors.append(name)
                    seen.add(key)

        # Fallback 1: walk author/name span classes (including hashed CSS classes).
        if not authors:
            author_spans = card.find_all("span", class_=re.compile(r"author", re.IGNORECASE))
            for author_span in author_spans:
                name_span = author_span.find("span", class_=re.compile(r"name", re.IGNORECASE))
                candidate = name_span.get_text(" ", strip=True) if name_span else author_span.get_text(" ", strip=True)
                name = clean_text(candidate)
                if not name:
                    continue
                if name.lower() in {"author", "authors", "share"}:
                    continue
                key = name.lower()
                if key not in seen:
                    authors.append(name)
                    seen.add(key)

        # Fallback 2: parse author display name from /author/<slug> links on the card.
        if not authors:
            author_links = card.select("a[href*='/author/']")
            for link in author_links:
                href = link.get("href") or ""
                m = re.search(r"/author/([^/?#]+)", href)
                if not m:
                    continue
                slug = m.group(1)
                slug = re.sub(r"-ncpn\d+$", "", slug)
                slug = slug.replace("-", " ").strip()
                name = clean_text(" ".join(part.capitalize() for part in slug.split()))
                if not name:
                    continue
                key = name.lower()
                if key not in seen:
                    authors.append(name)
                    seen.add(key)

        # Fallback 3: recursive JSON-LD lookup keyed by card id.
        if not authors and jmeta.get("authors"):
            for name in jmeta["authors"]:
                key = name.lower()
                if key not in seen:
                    authors.append(name)
                    seen.add(key)

        author_value = " | ".join(authors) if authors else pd.NA

        parsed.append(
            {
                "url": url,
                "topic": source_row.get("topic"),
                "title": main_title,
                "subtitle": subtitle if subtitle else pd.NA,
                "author": author_value,
                "datetime_posted": pubat if pubat else pd.NA,
                "label": source_row.get("label", "NBC News"),
                "card_id": card_id,
            }
        )

    return parsed

In [31]:
expanded_rows = []
page_logs = []

source_by_url = (
    live_source_rows.sort_values("url")
    .drop_duplicates(subset=["normalized_url"], keep="first")
    .to_dict("records")
)

for source in source_by_url:
    live_url = source["normalized_url"]
    try:
        card_rows = parse_live_blog_cards(source)
        expanded_rows.extend(card_rows)
        page_logs.append(
            {
                "normalized_url": live_url,
                "status": "ok",
                "cards_found": len(card_rows),
                "error": pd.NA,
            }
        )
    except Exception as exc:
        page_logs.append(
            {
                "normalized_url": live_url,
                "status": "error",
                "cards_found": 0,
                "error": str(exc),
            }
        )

cards_df = pd.DataFrame(expanded_rows)
page_log_df = pd.DataFrame(page_logs)

if not cards_df.empty and "card_id" not in cards_df.columns:
    cards_df["card_id"] = pd.NA

# Drop overflow-content cards (non-standard cards that add noise).
overflow_mask = (
    pd.Series(False, index=cards_df.index)
    if cards_df.empty or "card_id" not in cards_df.columns
    else cards_df["card_id"].fillna("").str.contains("overflow-content", na=False)
)
overflow_count = int(overflow_mask.sum())
cards_df = cards_df.loc[~overflow_mask].copy()

print(f"Live pages attempted: {len(page_log_df)}")
print(page_log_df["status"].value_counts(dropna=False))
print(f"Overflow cards dropped: {overflow_count}")
print(f"Expanded card rows after overflow drop: {len(cards_df)}")
cards_df.head()

Live pages attempted: 64
status
ok    64
Name: count, dtype: int64
Overflow cards dropped: 442
Expanded card rows after overflow drop: 3651


,url,topic,title,subtitle,author,datetime_posted,label,card_id
0,https://www.nbcnews.com/media/live-blog/-fox-n...,media,Fox News and Dominion reach settlement on firs...,Scoop: The special master investigation is ove...,Daniel Arkin,2023-04-19T00:39:07.836Z,NBC News,rcrd12612
1,https://www.nbcnews.com/media/live-blog/-fox-n...,media,Fox News and Dominion reach settlement on firs...,Dominion settlement ranks among biggest for U....,ZoÃ« Richards | Jane C. Timm,2023-04-18T23:44:36.165Z,NBC News,rcrd12610
2,https://www.nbcnews.com/media/live-blog/-fox-n...,media,Fox News and Dominion reach settlement on firs...,Bill O'Reilly on Dominion settlement: 'The nig...,Julia Jester,2023-04-18T23:23:22.087Z,NBC News,rcrd12611
3,https://www.nbcnews.com/media/live-blog/-fox-n...,media,Fox News and Dominion reach settlement on firs...,Dominion suggests no on-air apology coming fro...,Julia Jester,2023-04-18T23:02:37.125Z,NBC News,rcrd12609
4,https://www.nbcnews.com/media/live-blog/-fox-n...,media,Fox News and Dominion reach settlement on firs...,NaN,NBC News,2023-04-18T22:08:40.834Z,NBC News,rcrd12608


## 3. Rebuild the Dataset with Cleaned Live Cards

We remove the original 80 sparse live-update source rows and append extracted card-level rows.

This turns each live page into many usable observations and produces a repaired NBC dataset with improved completeness.

### Fallback priority (DOM then JSON-LD)

Extraction favors what the browser would render first, then fills gaps from recursive JSON-LD keyed by `#rcrd...`:
- Authors: markup → `/author/<slug>` → JSON-LD `author` objects
- Subtitle: card headline (`h2`) → JSON-LD `headline` when markup is missing
- Timestamp: card `data-pubat` → JSON-LD `datePublished` / `dateModified`

After merge we drop NBC rows that still cannot support analysis: broken titles, and live-update rows missing any of author, subtitle, or timestamp.

In [32]:
drop_mask = live_mask
nbc_without_live_nulls = nbc.loc[~drop_mask].copy()

cards_core = cards_df[["url", "topic", "title", "subtitle", "author", "datetime_posted", "label"]].copy()
nbc_backfilled = pd.concat([nbc_without_live_nulls, cards_core], ignore_index=True)

print(f"Rows before: {len(nbc)}")
print(f"Dropped source rows: {drop_mask.sum()}")
print(f"Card rows added: {len(cards_core)}")
print(f"Rows after merge: {len(nbc_backfilled)}")
print("NA counts after merge (before final clean):")
print(nbc_backfilled.isna().sum())

# Final clean-up for downstream modeling
broken_title_mask = nbc_backfilled["title"].isna()
broken_title_removed = int(broken_title_mask.sum())

nbc_work = nbc_backfilled.loc[~broken_title_mask].copy()
live_url_mask = nbc_work["url"].str.contains("live-updates", na=False)

live_incomplete_mask = live_url_mask & (
    nbc_work["author"].isna()
    | nbc_work["subtitle"].isna()
    | nbc_work["datetime_posted"].isna()
)
live_incomplete_removed = int(live_incomplete_mask.sum())

nbc_backfilled = nbc_work.loc[~live_incomplete_mask].copy()

print(f"Broken-title rows dropped: {broken_title_removed}")
print(
    "Incomplete live-update rows dropped "
    "(missing author, subtitle, or datetime_posted): "
    f"{live_incomplete_removed}"
)

nbc_mask = nbc_backfilled["url"].str.contains("nbcnews.com", na=False)
nbc_metadata_sparse = nbc_mask & (
    nbc_backfilled["author"].isna() | nbc_backfilled["datetime_posted"].isna()
)
nbc_metadata_removed = int(nbc_metadata_sparse.sum())
nbc_backfilled = nbc_backfilled.loc[~nbc_metadata_sparse].copy()

print(
    "Sparse NBC metadata rows dropped "
    "(missing author or datetime_posted on nbcnews.com): "
    f"{nbc_metadata_removed}"
)
print(f"Rows after final clean: {len(nbc_backfilled)}")
print("NA counts after final clean:")
print(nbc_backfilled.isna().sum())

Rows before: 1805
Dropped source rows: 80
Card rows added: 3651
Rows after: 5376
NA counts after backfill:
url                  0
topic                0
title                4
subtitle           205
author              16
datetime_posted     14
label                0
dtype: int64


## 4. QA and Triage of Remaining Gaps

After rebuilding, we audit what is still missing and surface the exact links/card ids for manual follow-up.

This keeps the workflow transparent: automated fill first, then targeted investigation for residual edge cases.

In [ ]:
# Rows that satisfy the live-complete gate (matching what merges into NBC).
_live = cards_df["url"].str.contains("live-updates", na=False)
_cards_effective = cards_df.loc[
    ~(_live &
      (
          cards_df["author"].isna()
          | cards_df["subtitle"].isna()
          | cards_df["datetime_posted"].isna()
      ))
].copy()

print(
    "Card rows omitted before merge (live + incomplete triple): ",
    len(cards_df) - len(_cards_effective),
)

# Use `_cards_effective` so QA matches what became rows in `nbc_backfilled`.
na_author_rows = _cards_effective[_cards_effective["author"].isna()].copy()
na_subtitle_rows = _cards_effective[_cards_effective["subtitle"].isna()].copy()
na_datetime_rows = _cards_effective[_cards_effective["datetime_posted"].isna()].copy()

print(f"Card rows with author NA (effective): {len(na_author_rows)}")
print(f"Card rows with subtitle NA (effective): {len(na_subtitle_rows)}")
print(f"Card rows with datetime NA (effective): {len(na_datetime_rows)}")

print("\nUnique URLs with author NA:", na_author_rows["url"].nunique())
print("Unique URLs with subtitle NA:", na_subtitle_rows["url"].nunique())
print("Unique URLs with datetime NA:", na_datetime_rows["url"].nunique())


Card rows with author NA: 442
Card rows with subtitle NA: 643
Card rows with datetime NA: 442

Unique URLs with author NA: 51
Unique URLs with subtitle NA: 62
Unique URLs with datetime NA: 51


In [24]:
# Inspect links still missing author (with card ids)
author_na_links = (
    na_author_rows[["url", "card_id", "title", "subtitle", "datetime_posted"]]
    .sort_values(["url", "card_id"], na_position="last")
    .reset_index(drop=True)
)

print("Top 30 rows with author NA:")
author_na_links.head(30)

# Clickable URL + card_id table for the first 30 rows (if IPython is available)
top30_clickable = author_na_links[["url", "card_id"]].head(30).copy()

try:
    from IPython.display import HTML, display

    top30_clickable["url"] = top30_clickable["url"].apply(
        lambda u: f'<a href="{u}" target="_blank">{u}</a>' if pd.notna(u) else ""
    )
    display(HTML(top30_clickable.to_html(escape=False, index=False)))
except ModuleNotFoundError:
    print("IPython not available in this runtime; showing plain table instead.")
    print(top30_clickable.to_string(index=False))

Top 30 rows with author NA:


url,card_id
https://www.nbcnews.com/news/live-blog/israel-hamas-war-live-updates-rcna135801,rcrd32132-overflow-content
https://www.nbcnews.com/news/live-blog/israel-hamas-war-live-updates-rcna135801,rcrd32135-overflow-content
https://www.nbcnews.com/news/live-blog/israel-hamas-war-live-updates-rcna135801,rcrd32153-overflow-content
https://www.nbcnews.com/news/live-blog/israel-hamas-war-live-updates-rcna135801,rcrd32164-overflow-content
https://www.nbcnews.com/news/weather/live-blog/hurricane-helene-live-updates-rcna173973,rcrd58517-overflow-content
https://www.nbcnews.com/news/weather/live-blog/hurricane-milton-live-updates-rcna174253,rcrd58959-overflow-content
https://www.nbcnews.com/news/world/live-blog/iran-attack-live-updates-rcna148136,rcrd39134-overflow-content
https://www.nbcnews.com/news/world/live-blog/iran-attack-live-updates-rcna148136,rcrd39154-overflow-content
https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-gazas-health-system-collapsing-battles-i-rcna128980,rcrd27775-overflow-content
https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-gazas-health-system-collapsing-battles-i-rcna128980,rcrd27786-overflow-content


In [25]:
# Compact unique URL lists for follow-up manual checks
author_na_unique_urls = sorted(na_author_rows["url"].dropna().unique())
subtitle_na_unique_urls = sorted(na_subtitle_rows["url"].dropna().unique())
datetime_na_unique_urls = sorted(na_datetime_rows["url"].dropna().unique())

print("Author NA unique URLs:")
for u in author_na_unique_urls:
    print(u)

print("\nSubtitle NA unique URLs:")
for u in subtitle_na_unique_urls:
    print(u)

print("\nDatetime NA unique URLs:")
for u in datetime_na_unique_urls:
    print(u)

Author NA unique URLs:
https://www.nbcnews.com/news/live-blog/israel-hamas-war-live-updates-rcna135801
https://www.nbcnews.com/news/weather/live-blog/hurricane-helene-live-updates-rcna173973
https://www.nbcnews.com/news/weather/live-blog/hurricane-milton-live-updates-rcna174253
https://www.nbcnews.com/news/world/live-blog/iran-attack-live-updates-rcna148136
https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-gazas-health-system-collapsing-battles-i-rcna128980
https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-netanyahu-vows-nothing-will-stop-us-rcna129682
https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-rcna126500
https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-rcna128049
https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-rcna132789
https://www.nbcnews.com/news/world/live-blog/israel-hamas-war-live-updates-rcna134047
https://www.nbcnews.com/news/world/live-blog/is

### Interpreting Remaining NAs

Residual missing values are expected for some live updates (for example, cards without explicit bylines or cards that function as visual/layout separators).

The investigative tables below are the manual triage queue for deciding whether remaining rows should be enriched further, retained as-is, or excluded.

## 5. Persist Artifacts

We save:
- page-level parse diagnostics,
- extracted card-level records,
- final repaired NBC dataset used downstream.

In [12]:
page_log_df.to_csv("../data/interim/nbc_live_updates_page_parse_log.csv", index=False)
cards_df.to_csv("../data/interim/nbc_live_updates_cards_extracted.csv", index=False)
nbc_backfilled.to_csv("../data/raw/nbc_scraped_all_back.csv", index=False)

print("Wrote:")
print("- ../data/interim/nbc_live_updates_page_parse_log.csv")
print("- ../data/interim/nbc_live_updates_cards_extracted.csv")
print("- ../data/raw/nbc_scraped_all_back.csv")

Wrote:
- ../data/interim/nbc_live_updates_page_parse_log.csv
- ../data/interim/nbc_live_updates_cards_extracted.csv
- ../data/raw/nbc_scraped_all_back.csv


## 6. Project Wrap-Up and Next Steps

### What this notebook accomplished
- Scoped and repaired NBC live-update records identified in exploration.
- Replaced sparse live source rows with card-level observations.
- Improved field completeness using layered extraction and overflow-card filtering.

### What remains
- Decide whether omitted sparse cards merit custom parsing versus staying excluded.
- Document any scraping policy tweaks if NBC changes markup again later.

### Output Contract

This notebook intentionally writes three artifacts:
- page parse log (run diagnostics),
- card-level extraction table (intermediate audit layer),
- repaired NBC dataset (downstream modeling input).

Keeping all three makes the pipeline reproducible and debuggable.